In [ ]:
from Tools_U1_plaquette import *

In [ ]:
# ---------------- Parameters ----------------
λ_m=1.0
#λ_m=np.inf

Ns = np.arange(3, 15, 2)
N_θ = 10001

if λ_m == np.inf:
    start = 1e-1        
    stop = 8e2          
    λ_ovr_g = np.logspace(np.log10(start), np.log10(stop), num=20)
else:
    start = 1e-1        
    stop = 1e3        
    λ_ovr_g = np.logspace(np.log10(start), np.log10(stop), num=40)

# ---------------- Storage ----------------
expval_Pop_0 = np.zeros((len(Ns), len(λ_ovr_g)))
expval_n_sqr_0 = np.zeros((len(Ns), len(λ_ovr_g)))

# ---------------- Loop ----------------
for i, n in enumerate(Ns):
    print("Computations for N =", n)
    for j, λ_g in enumerate(λ_ovr_g):
        print("Computing for λ/g =", λ_g)
        E_ν_vector, exp_plusiθ_op_matrix, exp_minusiθ_op_matrix, nθ_op_matrix, nθ_sqr_op_matrix = single_link_data(λ_m, λ_g, n, N_θ)
        """
        H = H_0 + H_additional

        H_0 = Σ_l H_l, with H_l = Σ_l [ 2/(λ/m) + 1/(λ/g) ] n_l^2 - cos(θ_l), l={12,24,13,34} 
        have analytical solution obdatied in single_link_data(), so in Mathieu basis are 
        diagonal with same eigenvalues E_ν_vector for each link.

        H_additional = 2/(λ/m) (n12 * n13 + n24 * n34 - n12 * n24 - n13 * n34 )
        """
        H_0_l = sp.diags(E_ν_vector, offsets = 0, format='csr')
        id_N = sp.eye(n, format='csr')
        # Choice of order is important for Kronecker products. Here we take [n12, n24, n13, n34]
        H_0 = (
              kron_n(H_0_l, id_N, id_N, id_N) 
            + kron_n(id_N, H_0_l, id_N, id_N) 
            + kron_n(id_N, id_N, H_0_l, id_N) 
            + kron_n(id_N, id_N, id_N, H_0_l)
        )
        H_additional = (2.0/λ_m) * (
            kron_n(nθ_op_matrix, id_N, nθ_op_matrix, id_N) # n12 * n13
          + kron_n(id_N, nθ_op_matrix, id_N, nθ_op_matrix) # n24 * n34
          - kron_n(nθ_op_matrix, nθ_op_matrix, id_N, id_N) # n12 * n24
          - kron_n(id_N, id_N, nθ_op_matrix, nθ_op_matrix) # n13 * n34
        )
        H_mathieu = H_0 + H_additional
        
        is_hermitian = (H_mathieu - H_mathieu.getH()).nnz == 0
        print("Hermitian?", is_hermitian)
        assert sp.isspmatrix(H_mathieu)
        off_diag_nnz_mathieu = (H_mathieu - sp.diags(H_mathieu.diagonal(), format="csr")).nnz
        print("Off-diagonal elements:", off_diag_nnz_mathieu)
        print("Shape of the Hamiltonian H_mathieu -> ", H_mathieu.shape)
        
        E, ψ = spla.eigsh(H_mathieu, k=1, which="SA", tol=1e-8, maxiter=100000)
        ψ_chosen = ψ[:, 0]

        P = 0.5 * (
            kron_n(exp_plusiθ_op_matrix, exp_plusiθ_op_matrix, exp_minusiθ_op_matrix, exp_minusiθ_op_matrix) +
            kron_n(exp_minusiθ_op_matrix, exp_minusiθ_op_matrix, exp_plusiθ_op_matrix, exp_plusiθ_op_matrix)
        )
        n_sqr = kron_n(nθ_sqr_op_matrix, id_N, id_N, id_N)

        expval_Pop_0[i, j] =  np.vdot(ψ_chosen, P @ ψ_chosen).real
        expval_n_sqr_0[i, j] =  np.vdot(ψ_chosen, n_sqr @ ψ_chosen).real

In [ ]:
# Save results
if λ_m == np.inf:
    filename_m0 = f"LGT_SCQs_plaquette_reduced_def_OCT25/LGT_SCQs_plaquette_reduced4vars_alphas0_plaquetteop_nsqr_mathieu_m0comparison_draft.npz"
    np.savez_compressed(filename_m0,
                        Ns_m0=Ns,
                        λ_ovr_g_m0=λ_ovr_g,
                        results_n_sqr_charge_0_m0=expval_n_sqr_0,
                        results_Pop_charge_0_m0=expval_Pop_0
    )
    print(f"\nResults saved to {filename_m0}")
else:
    filename_mneq0 = f"LGT_SCQs_plaquette_reduced_def_OCT25/LGT_SCQs_plaquette_reduced4vars_alphas0_plaquetteop_nsqr_mathieu_mneq0_λ_m_{λ_m}_comparison_draft.npz"
    np.savez_compressed(filename_mneq0,
                        Ns_mneq0=Ns,
                        λ_ovr_g_mneq0=λ_ovr_g,
                        results_n_sqr_charge_0_mneq0=expval_n_sqr_0,
                        results_Pop_charge_0_mneq0=expval_Pop_0
    )
    print(f"\nResults saved to {filename_mneq0}")